In [ ]:
import torch
print(torch.__version__)

2.10.0+cu128


In [ ]:
import torch
print(torch.cuda.is_available())

True


In [ ]:
!pip install kagglehub pandas --quiet

import kagglehub
import os
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")
file_path = os.path.join(path, "IMDB Dataset.csv")

df = pd.read_csv(file_path)

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.


In [ ]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    return text

df['review'] = df['review'].apply(clean_text)

df = df.sample(2000, random_state=42)

In [ ]:
sentences = []
tags = []

for _, row in df.iterrows():
    words = row['review'].split()[:20]
    label = row['sentiment']

    if label == "positive":
        tag_seq = ["POS"] * len(words)
    else:
        tag_seq = ["NEG"] * len(words)

    sentences.append(words)
    tags.append(tag_seq)


In [ ]:
word_to_ix = {}
tag_to_ix = {"POS": 0, "NEG": 1}

for sentence in sentences:
    for word in sentence:
        if word not in word_to_ix:
            word_to_ix[word] = len(word_to_ix)

In [ ]:
def prepare_sequence(seq, to_ix):
    return torch.tensor([to_ix.get(w, 0) for w in seq], dtype=torch.long).to(device)

In [ ]:
class LSTMTagger(nn.Module):
    def __init__(self, vocab_size, tagset_size, embedding_dim=100, hidden_dim=128):
        super(LSTMTagger, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, tagset_size)

    def forward(self, sentence):
        embeds = self.embedding(sentence)
        lstm_out, _ = self.lstm(embeds.view(len(sentence), 1, -1))
        tag_space = self.fc(lstm_out.view(len(sentence), -1))
        return torch.log_softmax(tag_space, dim=1)

In [ ]:
model = LSTMTagger(len(word_to_ix), len(tag_to_ix)).to(device)

loss_function = nn.NLLLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
for epoch in range(10):
    total_loss = 0

    for sentence, tag_seq in zip(sentences, tags):
        model.zero_grad()

        sentence_in = prepare_sequence(sentence, word_to_ix)
        targets = torch.tensor(
            [tag_to_ix[t] for t in tag_seq], dtype=torch.long
        ).to(device)

        tag_scores = model(sentence_in)

        loss = loss_function(tag_scores, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

print("Training complete!")

Epoch 1, Loss: 1374.8916
Epoch 2, Loss: 1149.6776
Epoch 3, Loss: 714.0509
Epoch 4, Loss: 337.7332
Epoch 5, Loss: 141.9046
Epoch 6, Loss: 73.3890
Epoch 7, Loss: 36.4493
Epoch 8, Loss: 19.2433
Epoch 9, Loss: 25.0577
Epoch 10, Loss: 9.6846
Training complete!


In [ ]:
ix_to_tag = {v: k for k, v in tag_to_ix.items()}

def predict(sentence):
    with torch.no_grad():
        words = sentence.lower().split()
        inputs = prepare_sequence(words, word_to_ix)
        scores = model(inputs)
        predicted = torch.argmax(scores, dim=1)

        tags = [ix_to_tag[i.item()] for i in predicted]

        print("\nSentence:", sentence)
        print("Predicted Tags:", tags)

In [ ]:
predict("this movie is amazing and wonderful")
predict("this film was terrible and boring")


Sentence: this movie is amazing and wonderful
Predicted Tags: ['POS', 'POS', 'POS', 'POS', 'POS', 'POS']

Sentence: this film was terrible and boring
Predicted Tags: ['NEG', 'NEG', 'NEG', 'NEG', 'NEG', 'NEG']


In [ ]:
def calculate_accuracy():
    correct = 0
    total = 0

    with torch.no_grad():
        for sentence, tag_seq in zip(sentences, tags):
            inputs = prepare_sequence(sentence, word_to_ix)
            scores = model(inputs)
            predicted = torch.argmax(scores, dim=1)

            targets = torch.tensor([tag_to_ix[t] for t in tag_seq]).to(device)

            correct += (predicted == targets).sum().item()
            total += len(targets)

    print("Token Accuracy:", correct / total)

calculate_accuracy()

Token Accuracy: 0.9880997024925623


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

def evaluate_model():
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for sentence, tag_seq in zip(sentences, tags):
            inputs = prepare_sequence(sentence, word_to_ix)
            scores = model(inputs)
            predicted = torch.argmax(scores, dim=1).cpu().numpy()

            targets = [tag_to_ix[t] for t in tag_seq]

            all_preds.extend(predicted)
            all_targets.extend(targets)

    precision = precision_score(all_targets, all_preds, average='weighted')
    recall = recall_score(all_targets, all_preds, average='weighted')
    f1 = f1_score(all_targets, all_preds, average='weighted')

    print("\nEvaluation Metrics:")
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1-score:", f1)

evaluate_model()


Evaluation Metrics:
Precision: 0.9883608893848468
Recall: 0.9880997024925623
F1-score: 0.9880947287593517


In [ ]:
from sklearn.metrics import classification_report

# Re-calculate all_preds and all_targets to make them accessible
all_preds = []
all_targets = []

with torch.no_grad():
    for sentence, tag_seq in zip(sentences, tags):
        inputs = prepare_sequence(sentence, word_to_ix)
        scores = model(inputs)
        predicted = torch.argmax(scores, dim=1).cpu().numpy()

        targets = [tag_to_ix[t] for t in tag_seq]

        all_preds.extend(predicted)
        all_targets.extend(targets)

print(classification_report(all_targets, all_preds, target_names=list(tag_to_ix.keys())))

              precision    recall  f1-score   support

         POS       1.00      0.98      0.99     19520
         NEG       0.98      1.00      0.99     20479

    accuracy                           0.99     39999
   macro avg       0.99      0.99      0.99     39999
weighted avg       0.99      0.99      0.99     39999

